In [1]:
import io
import unicodedata
 

In [2]:
import numpy as np
import pandas as pd

In [3]:
INPUT_PATH = "diagnosticos_itaca.csv"
OUTPUT_PATH = "diagnosticos_itaca_clean.csv"
 

In [4]:
SOURCE_ENCODING = "latin-1"
ROW_PREFIX = "DIAG"  # prefijo de id_diagnostico,

In [5]:
EXPECTED_COLUMNS = [
    "id_diagnostico",
    "sector",
    "tamano_empresa",
    "porcentaje_procesos_documentados",
    "presupuesto_anual_tecnologia",  # nombre normalizado (sin tilde)
    "respuesta_texto",
    "nivel_madurez",
    "recomendacion_principal",
]

In [6]:
EXPECTED_SECTORS = {"Tecnologia", "Manufactura", "Retail", "Servicios"}
EXPECTED_SIZES = {"Micro", "Pequena", "Mediana", "Grande"}
EXPECTED_LEVELS = {"Inicial", "En Desarrollo", "Definido", "Optimizado"}
 

In [7]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
 

In [8]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

## EXTRACCION - lectura cruda con encoding correcto

In [9]:
with open(INPUT_PATH, "rb") as f:
    raw_bytes = f.read()
 

In [10]:
text = raw_bytes.decode(SOURCE_ENCODING)

### Normalizar terminadores CRLF -> LF

In [11]:
text = text.replace("\r\n", "\n")
lines = text.strip().split("\n")

In [12]:
header = lines[0]
data_lines = lines[1:]

In [13]:
print(f"Lineas totales (incluyendo header): {len(lines)}")
print(f"Filas de datos: {len(data_lines)}")
print(f"Header crudo: {header}")

Lineas totales (incluyendo header): 3001
Filas de datos: 3000
Header crudo: id_diagnostico,sector,tamano_empresa,porcentaje_procesos_documentados,presupuesto_anual_tecnología,respuesta_texto,nivel_madurez,recomendacion_principal


## REPARACION ESTRUCTURAL - quoting malformado

Patron del defecto: la linea entera envuelta en comillas dobles y las
comillas internas duplicadas, p.ej.:
  "DIAG_0000,Tecnologia,Micro,0.05,3000000,""texto, con comas"",Inicial,Recomendacion"

In [14]:
# Reparacion: quitar comillas exteriores y convertir "" -> "
# Las lineas sanas se dejan intactas.

In [15]:
def repair_line(line: str) -> tuple[str, bool]:
    """Repara una linea con quoting malformado. Devuelve (linea, fue_reparada)."""
    if line.startswith(f'"{ROW_PREFIX}') and line.endswith('"'):
        inner = line[1:-1].replace('""', '"')
        return inner, True
    return line, False

In [16]:
repaired_lines = []
n_repaired = 0

In [17]:
for line in data_lines:
    fixed, was_repaired = repair_line(line)
    repaired_lines.append(fixed)
    n_repaired += int(was_repaired)
 

In [18]:
pct = 100.0 * n_repaired / len(data_lines)
print(f"Filas reparadas (quoting malformado): {n_repaired} ({pct:.1f}%)")
 

Filas reparadas (quoting malformado): 920 (30.7%)


## PARSEO A DATAFRAME

In [22]:
csv_text = "\n".join([header] + repaired_lines)
df = pd.read_csv(io.StringIO(csv_text))

In [23]:
print(f"Shape: {df.shape}")
print(f"Columnas originales: {list(df.columns)}")
 

Shape: (3000, 8)
Columnas originales: ['id_diagnostico', 'sector', 'tamano_empresa', 'porcentaje_procesos_documentados', 'presupuesto_anual_tecnología', 'respuesta_texto', 'nivel_madurez', 'recomendacion_principal']


## NORMALIZACION DE ESQUEMA

Nombres de columnas: minusculas, sin tildes (evita problemas de
 portabilidad entre entornos con distinto locale/encoding).

In [25]:
def strip_accents(s: str) -> str:
    return "".join(
        c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn"
    )
 

In [26]:
df.columns = [strip_accents(c.strip().lower()) for c in df.columns]

Verificar strip accent function 

In [29]:
assert list(df.columns) == EXPECTED_COLUMNS, (
    f"Esquema inesperado.\nEsperado: {EXPECTED_COLUMNS}\nObtenido: {list(df.columns)}"
)
print("Esquema validado: 8 columnas con nombres normalizados.")

Esquema validado: 8 columnas con nombres normalizados.


##  Normalizacion de valores categoricos: quitar tildes y espacios

In [31]:
for col in ["sector", "tamano_empresa", "nivel_madurez"]:
    df[col] = df[col].astype(str).str.strip().map(strip_accents)
 

In [34]:
# typos
df["porcentaje_procesos_documentados"] = pd.to_numeric(
    df["porcentaje_procesos_documentados"], errors="coerce"
)

In [35]:
df["presupuesto_anual_tecnologia"] = pd.to_numeric(
    df["presupuesto_anual_tecnologia"], errors="coerce"
)

In [36]:
df["respuesta_texto"] = df["respuesta_texto"].astype(str).str.strip()
 

In [37]:
print(df.dtypes)

id_diagnostico                       object
sector                               object
tamano_empresa                       object
porcentaje_procesos_documentados    float64
presupuesto_anual_tecnologia          int64
respuesta_texto                      object
nivel_madurez                        object
recomendacion_principal              object
dtype: object


In [38]:
df.head()

,id_diagnostico,sector,tamano_empresa,porcentaje_procesos_documentados,presupuesto_anual_tecnologia,respuesta_texto,nivel_madurez,recomendacion_principal
0,DIAG_0000,Tecnologia,Micro,0.05,3000000,"El trabajo es muy empírico, no hay documentaci...",Inicial,Mapear flujos de trabajo de desarrollo y usar ...
1,DIAG_0001,Tecnologia,Grande,0.81,261000000,"Todo está automatizado, usamos datos para mejo...",Optimizado,Aplicar MLOps y analítica predictiva para opti...
2,DIAG_0002,Tecnologia,Pequena,0.27,47000000,Hay guías básicas pero no siempre se cumplen l...,En Desarrollo,Integrar herramientas de tickets con repositor...
3,DIAG_0003,Tecnologia,Pequena,0.30,19000000,Hay guías básicas pero no siempre se cumplen l...,En Desarrollo,Integrar herramientas de tickets con repositor...
4,DIAG_0004,Manufactura,Micro,0.08,6000000,"Todo es manual, dependemos de una sola persona...",Inicial,Estandarizar el registro diario de mermas e in...


In [44]:
df.describe()

,porcentaje_procesos_documentados,presupuesto_anual_tecnologia
count,3000.000000,3.000000e+03
mean,0.444657,8.352767e+07
std,0.279736,1.096172e+08
min,0.000000,1.000000e+06
25%,0.190000,1.000000e+07
50%,0.430000,4.100000e+07
75%,0.670000,1.102500e+08
max,1.000000,5.000000e+08


In [41]:
df.shape

(3000, 8)

## VALIDACION DE INTEGRIDAD

In [42]:
report = {}

In [45]:
report["n_rows"] = len(df)
report["nulls_total"] = int(df.isna().sum().sum())
report["duplicated_rows"] = int(df.duplicated().sum())
report["duplicated_ids"] = int(df["id_diagnostico"].duplicated().sum())
 

In [46]:
unexpected_sectors = set(df["sector"].unique()) - EXPECTED_SECTORS
unexpected_sizes = set(df["tamano_empresa"].unique()) - EXPECTED_SIZES
unexpected_levels = set(df["nivel_madurez"].unique()) - EXPECTED_LEVELS

In [47]:
report["unexpected_sectors"] = sorted(unexpected_sectors)
report["unexpected_sizes"] = sorted(unexpected_sizes)
report["unexpected_levels"] = sorted(unexpected_levels)

In [48]:
out_of_range_pct = df[
    (df["porcentaje_procesos_documentados"] < 0)
    | (df["porcentaje_procesos_documentados"] > 1)
]

In [49]:
report["pct_out_of_range"] = len(out_of_range_pct)

In [50]:
non_positive_budget = df[df["presupuesto_anual_tecnologia"] <= 0]
report["budget_non_positive"] = len(non_positive_budget)

## Reporte Claidad

In [52]:

print("REPORTE DE CALIDAD DE DATOS")
print("-" * 50)
for k, v in report.items():
    print(f"{k:>25}: {v}")
 
hard_fail = (
    report["nulls_total"] > 0
    or report["duplicated_rows"] > 0
    or report["duplicated_ids"] > 0
    or report["unexpected_levels"]
    or report["pct_out_of_range"] > 0
)
if hard_fail:
    raise ValueError("Validacion de integridad fallida. Revisar el reporte anterior.")
print("\nValidacion de integridad: OK")


REPORTE DE CALIDAD DE DATOS
--------------------------------------------------
                   n_rows: 3000
              nulls_total: 0
          duplicated_rows: 0
           duplicated_ids: 0
       unexpected_sectors: []
         unexpected_sizes: []
        unexpected_levels: []
         pct_out_of_range: 0
      budget_non_positive: 0

Validacion de integridad: OK


## EDA - DISTRIBUCIONES CATEGORICAS Y OBJETIVO

In [53]:
for col in ["sector", "tamano_empresa", "nivel_madurez"]:
    print(f"\n--- {col} ---")
    counts = df[col].value_counts()
    pcts = (100 * counts / len(df)).round(1)
    print(pd.DataFrame({"count": counts, "pct": pcts}))


--- sector ---
             count   pct
sector                  
Tecnologia     797  26.6
Manufactura    755  25.2
Retail         724  24.1
Servicios      724  24.1

--- tamano_empresa ---
                count   pct
tamano_empresa             
Grande            779  26.0
Pequena           744  24.8
Mediana           741  24.7
Micro             736  24.5

--- nivel_madurez ---
               count   pct
nivel_madurez             
En Desarrollo    923  30.8
Definido         911  30.4
Inicial          783  26.1
Optimizado       383  12.8


In [56]:
print("\nCrosstab sector x nivel_madurez ")
print(pd.crosstab(df["sector"], df["nivel_madurez"]))


Crosstab sector x nivel_madurez 
nivel_madurez  Definido  En Desarrollo  Inicial  Optimizado
sector                                                     
Manufactura         236            241      192          86
Retail              225            208      200          91
Servicios           211            229      184         100
Tecnologia          239            245      207         106


In [57]:
print("\n Crosstab tamano_empresa x nivel_madurez ")

print(pd.crosstab(df["tamano_empresa"], df["nivel_madurez"]))


 Crosstab tamano_empresa x nivel_madurez 
nivel_madurez   Definido  En Desarrollo  Inicial  Optimizado
tamano_empresa                                              
Grande               297            117       42         323
Mediana              395            201       85          60
Micro                 74            228      434           0
Pequena              145            377      222           0


## EDA - VARIABLES NUMERICAS

In [64]:
num_cols = ["porcentaje_procesos_documentados", "presupuesto_anual_tecnologia"]

df[num_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
porcentaje_procesos_documentados,3000.0,4.446567e-01,2.797361e-01,0.0,0.19,0.43,6.700000e-01,1.0
presupuesto_anual_tecnologia,3000.0,8.352767e+07,1.096172e+08,1000000.0,10000000.00,41000000.00,1.102500e+08,500000000.0


In [63]:
print("\n Estadisticas numericas por nivel_madurez ")

df.groupby("nivel_madurez")[num_cols].agg(["mean", "std", "min", "max"]).round(3)

 


 Estadisticas numericas por nivel_madurez 


porcentaje_procesos_documentados                  presupuesto_anual_tecnologia                                    
                                          mean    std  min  max                         mean           std        min        max
nivel_madurez                                                                                                                   
Definido                                 0.649  0.088  0.5  0.8                 1.008738e+08  2.841534e+07   50000000  150000000
En Desarrollo                            0.345  0.087  0.2  0.5                 2.987974e+07  1.189249e+07   10000000   50000000
Inicial                                  0.103  0.057  0.0  0.2                 5.593870e+06  2.928572e+06    1000000   10000000
Optimizado                               0.897  0.057  0.8  1.0                 3.308825e+08  1.028897e+08  150000000  500000000

In [66]:
print("\n  Correlacion (Spearman)  ")
df[num_cols].corr(method="spearman").round(3)


  Correlacion (Spearman)  


,porcentaje_procesos_documentados,presupuesto_anual_tecnologia
porcentaje_procesos_documentados,1.000,0.923
presupuesto_anual_tecnologia,0.923,1.000


## EDA - ANALISIS DE SEPARABILIDAD DEL OBJETIVO

Hallazgo clave del diagnostico inicial: nivel_madurez sigue bandas casi
deterministas de porcentaje_procesos_documentado

In [67]:
def band_rule(p: float) -> str:
    
    if p < 0.20:
        
        return "Inicial"
        
    if p < 0.50:
        
        return "En Desarrollo"
        
    if p < 0.80:
        
        return "Definido"
        
    return "Optimizado"


In [68]:
df["_band_pred"] = df["porcentaje_procesos_documentados"].apply(band_rule)

match_rate = (df["_band_pred"] == df["nivel_madurez"]).mean()

print(f"Coincidencia regla-de-bandas vs etiqueta real: {match_rate:.4f}")

Coincidencia regla-de-bandas vs etiqueta real: 0.9817


In [71]:
mismatches = df[df["_band_pred"] != df["nivel_madurez"]]

print(f"Registros que NO siguen la banda: {len(mismatches)}")

print("\nDistribucion de porcentaje en los mismatches (valores frontera):")

mismatches["porcentaje_procesos_documentados"].value_counts()

Registros que NO siguen la banda: 55

Distribucion de porcentaje en los mismatches (valores frontera):


porcentaje_procesos_documentados
0.2    23
0.8    19
0.5    13
Name: count, dtype: int64

In [72]:
df = df.drop(columns=["_band_pred"])

## EDA - ANALISIS DEL TEXTO (rama NLP)

In [73]:
n_unique_texts = df["respuesta_texto"].nunique()

print(f"Textos unicos en respuesta_texto: {n_unique_texts}")

Textos unicos en respuesta_texto: 32


In [75]:
lengths = df["respuesta_texto"].str.len()
print("\nLongitud del texto (caracteres):")
lengths.describe().round(1)


Longitud del texto (caracteres):


count    3000.0
mean       71.4
std        12.4
min        60.0
25%        63.0
50%        67.0
75%        73.0
max       102.0
Name: respuesta_texto, dtype: float64

In [76]:
word_counts = df["respuesta_texto"].str.split().str.len()

In [77]:
print("\nLongitud del texto (palabras):")
print(word_counts.describe().round(1))
print(f"\nMaximo de palabras: {int(word_counts.max())}  "
      f"-> guia para output_sequence_length de TextVectorization")
 


Longitud del texto (palabras):
count    3000.0
mean       10.8
std         2.1
min         8.0
25%         9.0
50%        10.0
75%        12.0
max        16.0
Name: respuesta_texto, dtype: float64

Maximo de palabras: 16  -> guia para output_sequence_length de TextVectorization


In [78]:
vocab = set()

In [80]:
for t in df["respuesta_texto"].unique():
    vocab.update(t.lower().split())

 

In [81]:
print(f"Vocabulario aproximado (tokens unicos por espacio): {len(vocab)}  "
      f"-> guia para max_tokens de TextVectorization")

Vocabulario aproximado (tokens unicos por espacio): 108  -> guia para max_tokens de TextVectorization


In [83]:
print("\n--- Textos unicos por nivel_madurez ---")
df.groupby("nivel_madurez")["respuesta_texto"].nunique()
 


--- Textos unicos por nivel_madurez ---


nivel_madurez
Definido         8
En Desarrollo    8
Inicial          8
Optimizado       8
Name: respuesta_texto, dtype: int64

In [85]:
print("\n Consistencia texto -> nivel")

texts_per_level = df.groupby("respuesta_texto")["nivel_madurez"].nunique()

ambiguous = texts_per_level[texts_per_level > 1]

print(f"Textos asociados a mas de un nivel: {len(ambiguous)} de {n_unique_texts}")
 


 Consistencia texto -> nivel
Textos asociados a mas de un nivel: 0 de 32


## EDA - CATALOGO DE RECOMENDACIONES

recomendacion_principal es funcion determinista de (sector, nivel_madurez).
Este bloque extrae el catalogo que alimentara el motor de recomendaciones
del backend (lookup table), y valida que la relacion sea 1 a 1.

In [86]:
catalog = (
    df.groupby(["sector", "nivel_madurez"])["recomendacion_principal"]
    .agg(["nunique", "first"])
    .rename(columns={"nunique": "n_recomendaciones", "first": "recomendacion"})
)

In [87]:
catalog

n_recomendaciones                                      recomendacion
sector      nivel_madurez                                                                      
Manufactura Definido                       1  Conectar los datos de producción en planta con...
            En Desarrollo                  1  Implementar un sistema MRP básico para la plan...
            Inicial                        1  Estandarizar el registro diario de mermas e in...
            Optimizado                     1  Implementar sensores IoT en maquinaria para ma...
Retail      Definido                       1  Integrar el ERP de inventarios con el CRM para...
            En Desarrollo                  1  Centralizar el inventario de todas las sucursa...
            Inicial                        1  Implementar un sistema POS básico para abandon...
            Optimizado                     1  Automatizar pronóstico de demanda con modelos ...
Servicios   Definido                       1  Automatizar la facturación y las encuestas de ...
            En Desarrollo                  1  Implementar un CRM para el seguimiento estruct...
            Inicial                        1  Digitalizar la base de datos de clientes y cen...
            Optimizado                     1  Utilizar analítica avanzada para hiper-persona...
Tecnologia  Definido                       1  Implementar CI/CD y pruebas automatizadas para...
            En Desarrollo                  1  Integrar herramientas de tickets con repositor...
            Inicial                        1  Mapear flujos de trabajo de desarrollo y usar ...
            Optimizado                     1  Aplicar MLOps y analítica predictiva para opti...

In [88]:
assert (catalog["n_recomendaciones"] == 1).all(), (
    "Se esperaba exactamente 1 recomendacion por combinacion (sector, nivel)."
)

In [89]:
n_combos = len(catalog)

In [90]:
print(f"\nCatalogo validado: {n_combos} combinaciones, 1 recomendacion por combo.")
 


Catalogo validado: 16 combinaciones, 1 recomendacion por combo.


In [91]:
# Exportar catalogo para el backend (motor de recomendaciones)
catalog_df = catalog.reset_index()[["sector", "nivel_madurez", "recomendacion"]]
catalog_df.to_csv("catalogo_recomendaciones.csv", index=False, encoding="utf-8")
print("Catalogo exportado: catalogo_recomendaciones.csv")
 

Catalogo exportado: catalogo_recomendaciones.csv


## CARGA - DATASET LIMPIO

In [93]:
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

In [94]:
print(f"Dataset limpio guardado: {OUTPUT_PATH}")
print(f"Shape final: {df.shape}")
print(f"Encoding de salida: UTF-8")

Dataset limpio guardado: diagnosticos_itaca_clean.csv
Shape final: (3000, 8)
Encoding de salida: UTF-8


In [95]:
df.head(5)

,id_diagnostico,sector,tamano_empresa,porcentaje_procesos_documentados,presupuesto_anual_tecnologia,respuesta_texto,nivel_madurez,recomendacion_principal
0,DIAG_0000,Tecnologia,Micro,0.05,3000000,"El trabajo es muy empírico, no hay documentaci...",Inicial,Mapear flujos de trabajo de desarrollo y usar ...
1,DIAG_0001,Tecnologia,Grande,0.81,261000000,"Todo está automatizado, usamos datos para mejo...",Optimizado,Aplicar MLOps y analítica predictiva para opti...
2,DIAG_0002,Tecnologia,Pequena,0.27,47000000,Hay guías básicas pero no siempre se cumplen l...,En Desarrollo,Integrar herramientas de tickets con repositor...
3,DIAG_0003,Tecnologia,Pequena,0.30,19000000,Hay guías básicas pero no siempre se cumplen l...,En Desarrollo,Integrar herramientas de tickets con repositor...
4,DIAG_0004,Manufactura,Micro,0.08,6000000,"Todo es manual, dependemos de una sola persona...",Inicial,Estandarizar el registro diario de mermas e in...


In [97]:
print("RESUMEN ETL STAGE 1 + EDA")
print("=" * 60)
print(f"Registros procesados:            {len(df)}")
print(f"Filas con quoting reparado:      {n_repaired} ({pct:.1f}%)")
print(f"Encoding corregido:              {SOURCE_ENCODING} -> utf-8")
print(f"Nulos / duplicados:              0 / 0")
print(f"Clases del objetivo:             4 (Optimizado minoritaria, ~13%)")
print(f"Textos unicos (rama NLP):        {n_unique_texts}")
print(f"Catalogo de recomendaciones:     {n_combos} combinaciones (sector x nivel)")
print("=" * 60)
 

RESUMEN ETL STAGE 1 + EDA
Registros procesados:            3000
Filas con quoting reparado:      920 (30.7%)
Encoding corregido:              latin-1 -> utf-8
Nulos / duplicados:              0 / 0
Clases del objetivo:             4 (Optimizado minoritaria, ~13%)
Textos unicos (rama NLP):        32
Catalogo de recomendaciones:     16 combinaciones (sector x nivel)
